# 🏷️ Stage 2 — Supervised Fact Detection with COOLANT × ViFactCheck

**Two-phase fine-tuning on ViFactCheck ground-truth labels (multimodal)**

📥 Input: `../processed_data/hdf5/vifactcheck_*.h5` (text [768,512], image [2048], labels)

🔗 Builds on: Stage 1 checkpoint (`training/checkpoints/best_model.pth`)

---

### Phases

| Phase  | Backbone                                    | Trainable               | Goal                               |
| ------ | ------------------------------------------- | ----------------------- | ---------------------------------- |
| **2a** | Frozen (`similarity_module`, `clip_module`) | `detection_module` only | Learn ViFactCheck label space fast |
| **2b** | Unfrozen                                    | All layers (low LR)     | End-to-end fine-tuning             |

### Label space

-   `0` = Supported (Real)
-   `1` = Refuted (Fake)
-   `2` = NEI (Not Enough Information)

> Set `NUM_CLASSES = 2` to collapse NEI into Supported (binary task)


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os
import math
import random
import json
import warnings
import datetime
from pathlib import Path
from collections import Counter

import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# ── Runtime / path resolution ─────────────────────────────────────────────────
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))


def detect_runtime() -> str:
    try:
        import google.colab
        from google.colab import drive

        drive.mount("/content/drive")
        return "colab"
    except ImportError:
        return "local"


def resolve_paths(root: Path):
    runtime = detect_runtime()
    if runtime == "colab":
        base = Path(
            os.environ.get(
                "COLAB_BASE_DIR",
                "/content/drive/MyDrive/Thesis_Final/fake-new-detection",
            )
        )
    else:
        base = Path(os.environ.get("LOCAL_BASE_DIR", "")) or root

    hdf5_dir = base / "notebooks" / "processed_data" / "hdf5"
    ckpt_dir = base / "training" / "checkpoints"
    save_dir = base / "training" / "checkpoints_stage2"
    for p in (hdf5_dir, ckpt_dir, save_dir):
        p.mkdir(parents=True, exist_ok=True)
    return runtime, base, hdf5_dir, ckpt_dir, save_dir


runtime, BASE_DIR, HDF5_DIR, CKPT_DIR, SAVE_DIR = resolve_paths(project_root)

# ── Device & seeds ────────────────────────────────────────────────────────────
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"🌐 Runtime : {runtime}")
print(f"🔧 Device  : {DEVICE}")
print(f"📂 HDF5    : {HDF5_DIR}")
print(f"💾 Stage1  : {CKPT_DIR}")
print(f"💾 Stage2  : {SAVE_DIR}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set NUM_CLASSES=2 for binary (Supported vs Refuted, drop NEI)
# Set NUM_CLASSES=3 for three-class (Supported / Refuted / NEI)
NUM_CLASSES = 3

IMAGE_DIM = 2048
TEXT_EMBED = 768
TEXT_SEQ_LEN = 512

CFG_2A = {
    # Architecture (must match Stage 1)
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,  # 64 (cross) + 16 + 16
    "h_dim": 64,
    "dropout": 0.3,
    # Phase-2a training
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "max_epochs": 20,
    "patience": 7,
    "seed": SEED,
    "device": str(DEVICE),
    "phase": "2a_frozen_backbone",
}

print("✅ Config ready")
print(
    f'   num_classes={NUM_CLASSES}, batch_size={CFG_2A["batch_size"]}, lr={CFG_2A["lr"]}'
)

## 📦 Dataset — load real ViFactCheck labels from HDF5


In [ ]:
# ── Copy HDF5 to local SSD on Colab ───────────────────────────────────────────
import shutil

if runtime == "colab":
    LOCAL_HDF5 = Path("/content/hdf5_local")
    LOCAL_HDF5.mkdir(exist_ok=True)
    for split in ["train", "dev", "test"]:
        src = HDF5_DIR / f"vifactcheck_{split}.h5"
        dst = LOCAL_HDF5 / f"vifactcheck_{split}.h5"
        if dst.exists():
            print(f"✅ {split}: already cached")
        elif src.exists():
            print(f"⏳ Copying {split} ({src.stat().st_size / 1024**3:.2f} GB)...")
            shutil.copy2(src, dst)
            print(f"✅ {split}: copied")
        else:
            print(f"❌ {split}: not found at {src}")
    HDF5_DIR = LOCAL_HDF5
    print(f"📂 HDF5_DIR → {HDF5_DIR} (local SSD)")
else:
    print(f"ℹ️  Local runtime — HDF5_DIR: {HDF5_DIR}")

In [ ]:
# ── HDF5Dataset with ground-truth labels ─────────────────────────────────────
class HDF5DatasetLabeled(Dataset):
    """
    Returns (text, image, label) from a ViFactCheck HDF5 file.

    text  : FloatTensor [768, 512]  — PhoBERT token embeddings
    image : FloatTensor [2048]      — ResNet50 features
    label : LongTensor  []          — ViFactCheck ground-truth label

    Args:
        num_classes: 2 → remap {0→0 (real), 1→1 (fake), 2→0 (NEI as real)}
                     3 → use labels as-is {0, 1, 2}
        drop_nei   : only used when num_classes=2; if True, samples with
                     label==2 are excluded entirely instead of remapped.
    """

    def __init__(self, hdf5_path: Path, num_classes: int = 3, drop_nei: bool = False):
        self.hdf5_path = hdf5_path
        self.num_classes = num_classes
        self._file = h5py.File(self.hdf5_path, "r")

        raw_labels = self._file["labels"][:].astype(np.int32)

        if num_classes == 2:
            if drop_nei:
                # Keep only label 0 (Supported) and 1 (Refuted)
                mask = raw_labels < 2
                self.indices = np.where(mask)[0]
            else:
                # NEI → 0 (treat as Supported/Real)
                raw_labels = np.where(raw_labels == 2, 0, raw_labels)
                self.indices = np.arange(len(raw_labels))
        else:
            self.indices = np.arange(len(raw_labels))

        self.labels = raw_labels
        print(
            f"✅ HDF5DatasetLabeled: {len(self.indices)} samples from {hdf5_path.name}"
        )
        dist = Counter(raw_labels[self.indices].tolist())
        label_names = {0: "Supported/Real", 1: "Refuted/Fake", 2: "NEI"}
        for k, v in sorted(dist.items()):
            print(
                f'   label {k} ({label_names.get(k, "?")}): {v} ({v/len(self.indices):.1%})'
            )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        text = (
            torch.from_numpy(self._file["text_features"][real_idx]).float().T
        )  # [768,512]
        image = torch.from_numpy(
            self._file["image_features"][real_idx]
        ).float()  # [2048]
        label = int(self.labels[real_idx])
        return text, image, torch.tensor(label, dtype=torch.long)

    def close(self):
        if hasattr(self, "_file") and self._file:
            self._file.close()
            self._file = None

    def __del__(self):
        self.close()


print("✅ HDF5DatasetLabeled class defined")

In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────────────────
BATCH_SIZE = CFG_2A["batch_size"]
DROP_NEI = False  # Set True to exclude NEI samples when NUM_CLASSES=2

datasets, loaders = {}, {}
for split in ["train", "dev", "test"]:
    h5_path = HDF5_DIR / f"vifactcheck_{split}.h5"
    if not h5_path.exists():
        raise FileNotFoundError(
            f"Missing {h5_path.name}. Run notebook 6_multimodal_hdf5_preprocessing.ipynb first."
        )
    print(f"\n── {split.upper()} ──")
    ds = HDF5DatasetLabeled(h5_path, num_classes=NUM_CLASSES, drop_nei=DROP_NEI)
    datasets[split] = ds
    loaders[split] = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )
    print(f"   {len(loaders[split])} batches")

# Smoke-test shape
text_s, img_s, lbl_s = datasets["train"][0]
print(f"\n📏 Sample shapes → text:{text_s.shape}  image:{img_s.shape}  label:{lbl_s}")

## 🧠 Model — load Stage 1 checkpoint + patch for arbitrary dims


In [ ]:
# ── Import COOLANT ────────────────────────────────────────────────────────────
for p in (str(BASE_DIR), str(BASE_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

import models.coolant_official as coolant_mod
from models.base import FastCNN

print("✅ COOLANT modules imported")

In [ ]:
# ── ResNetCOOLANT wrapper (same as Stage 1) ───────────────────────────────────
class ResNetCOOLANT(coolant_mod.COOLANT_Official):
    """COOLANT adapted for ResNet50 (2048-dim) and PhoBERT (768-dim)."""

    def encode_text(self, text):
        dummy = torch.zeros(text.size(0), IMAGE_DIM, device=text.device)
        t, _ = self.similarity_module.encoding(text, dummy)
        return t

    def encode_image(self, image):
        dummy = torch.zeros(
            image.size(0), TEXT_EMBED, TEXT_SEQ_LEN, device=image.device
        )
        _, i = self.similarity_module.encoding(dummy, image)
        return i

    def fuse_modalities(self, t, i):
        return torch.cat([t, i], dim=-1)


# ── Patch helpers (identical to Stage 1) ─────────────────────────────────────
def patch_encoding(enc):
    layers, done = [], False
    for l in enc.shared_image:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(IMAGE_DIM, l.out_features))
            done = True
        else:
            layers.append(l)
    enc.shared_image = nn.Sequential(*layers)


def patch_clip_projection(clip_module, is_image=True):
    target_dim = IMAGE_DIM if is_image else TEXT_EMBED
    proj = clip_module.image_projection if is_image else clip_module.text_projection
    layers, done = [], False
    for l in proj:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(target_dim, l.out_features))
            done = True
        else:
            layers.append(l)
    new_proj = nn.Sequential(*layers)
    if is_image:
        clip_module.image_projection = new_proj
    else:
        clip_module.text_projection = new_proj


def patch_unimodal(uni_module, prime_dim):
    shared_dim = 128
    for attr in ("text_uni", "image_uni"):
        setattr(
            uni_module,
            attr,
            nn.Sequential(
                nn.Linear(shared_dim, shared_dim),
                nn.BatchNorm1d(shared_dim),
                nn.ReLU(),
                nn.Linear(shared_dim, prime_dim),
                nn.BatchNorm1d(prime_dim),
                nn.ReLU(),
            ),
        )


def patch_fastcnn(m, input_dim, dropout=0.3):
    import types

    channel, kernels = 32, (1, 2, 4, 8)
    m.fast_cnn = nn.ModuleList(
        [
            nn.Sequential(
                nn.Conv1d(input_dim, channel, kernel_size=k),
                nn.BatchNorm1d(channel),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.AdaptiveMaxPool1d(1),
            )
            for k in kernels
        ]
    )

    def _fwd(self, x):
        return torch.cat([mod(x).squeeze(-1) for mod in self.fast_cnn], 1)

    m.forward = types.MethodType(_fwd, m)


print("✅ Patch helpers defined")

In [ ]:
# ── Build + patch model ───────────────────────────────────────────────────────
cfg_model = {
    k: CFG_2A[k]
    for k in (
        "shared_dim",
        "sim_dim",
        "clip_embed_dim",
        "feature_dim",
        "h_dim",
        "dropout",
    )
}
model = ResNetCOOLANT(cfg_model)

patch_encoding(model.similarity_module.encoding)
patch_encoding(model.detection_module.encoding)
patch_clip_projection(model.clip_module, is_image=True)
patch_clip_projection(model.clip_module, is_image=False)

prime_dim = (CFG_2A["feature_dim"] - 64) // 2  # = 16
patch_unimodal(model.detection_module.uni_repre, prime_dim)
patch_unimodal(model.detection_module.uni_se, 64)

for cnn_mod in (
    model.similarity_module.encoding.shared_text_encoding,
    model.detection_module.encoding.shared_text_encoding,
    model.detection_module.ambiguity_module.encoding.shared_text_encoding,
):
    patch_fastcnn(cnn_mod, TEXT_EMBED, CFG_2A["dropout"])

model = model.to(DEVICE)
print(
    f"✅ Model built and patched ({sum(p.numel() for p in model.parameters()):,} params)"
)

In [ ]:
# ── Replace detection classifier head for NUM_CLASSES ─────────────────────────
# COOLANT_Official detection_module.classifier expects 2 output classes by default.
# Re-wire only if using 3-class.
if NUM_CLASSES != 2:
    old_clf = model.detection_module.classifier
    in_features = (
        old_clf.in_features
        if hasattr(old_clf, "in_features")
        else old_clf[-1].in_features
    )
    if isinstance(old_clf, nn.Linear):
        model.detection_module.classifier = nn.Linear(in_features, NUM_CLASSES).to(
            DEVICE
        )
    else:
        # Sequential — replace last Linear
        layers = list(old_clf.children())
        layers[-1] = nn.Linear(layers[-1].in_features, NUM_CLASSES)
        model.detection_module.classifier = nn.Sequential(*layers).to(DEVICE)
    print(f"✅ Classifier head replaced → {NUM_CLASSES} classes")
else:
    print(f"ℹ️  Keeping original 2-class classifier head")

In [ ]:
# ── Load Stage 1 checkpoint ───────────────────────────────────────────────────
stage1_ckpt = CKPT_DIR / "best_model.pth"
if not stage1_ckpt.exists():
    raise FileNotFoundError(f"Stage 1 checkpoint not found: {stage1_ckpt}")

ckpt = torch.load(stage1_ckpt, map_location=DEVICE)
state = ckpt["model_state_dict"]

# Load with strict=False: classifier head may differ (2 vs 3 classes)
missing, unexpected = model.load_state_dict(state, strict=False)
print(
    f'✅ Stage 1 checkpoint loaded from epoch {ckpt.get("epoch", "?")}'
    f'  (val_loss={ckpt.get("val_loss", "?"):.4f})'
)
if missing:
    print(f"   Missing keys  ({len(missing)}): {missing[:5]}")
if unexpected:
    print(f"   Unexpected keys ({len(unexpected)}): {unexpected[:5]}")

## 🔒 Phase 2a — Frozen backbone, train detection head


In [ ]:
# ── Freeze similarity_module and clip_module ──────────────────────────────────
def set_requires_grad(module: nn.Module, value: bool):
    for p in module.parameters():
        p.requires_grad = value


def apply_phase(phase: str):
    """Toggle frozen / unfrozen state."""
    if phase == "2a":
        set_requires_grad(model.similarity_module, False)
        set_requires_grad(model.clip_module, False)
        set_requires_grad(model.detection_module, True)
        print("🔒 Phase 2a: backbone frozen, detection_module trainable")
    elif phase == "2b":
        set_requires_grad(model, True)  # unfreeze everything
        print("🔓 Phase 2b: all layers unfrozen")


apply_phase("2a")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
# ── Class weights — handle extreme label imbalance ────────────────────────────
train_labels_all = datasets["train"].labels[datasets["train"].indices]
class_counts = np.bincount(train_labels_all, minlength=NUM_CLASSES).astype(np.float64)
class_weights = len(train_labels_all) / (NUM_CLASSES * class_counts)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

_names = ["Supported/Real", "Refuted/Fake", "NEI"]
print("⚖️  Class weights (inverse-frequency):")
for i in range(NUM_CLASSES):
    print(
        f"   label {i} ({_names[i]}): n={int(class_counts[i])} ({class_counts[i]/len(train_labels_all):.1%}) → weight={class_weights[i]:.3f}"
    )

# ── Optimizer & scheduler for Phase 2a ───────────────────────────────────────
optim_2a = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG_2A["lr"],
    weight_decay=CFG_2A["weight_decay"],
)

steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CFG_2A["max_epochs"]
warmup_steps = max(1, int(0.05 * total_steps))  # 5% warmup

scheduler_2a = torch.optim.lr_scheduler.OneCycleLR(
    optim_2a,
    max_lr=CFG_2A["lr"],
    total_steps=total_steps,
    pct_start=warmup_steps / total_steps,
)

loss_ce = nn.CrossEntropyLoss(
    weight=class_weights_t, label_smoothing=CFG_2A["label_smoothing"]
)

print(f'\n✅ Optimizer ready  (lr={CFG_2A["lr"]}, wd={CFG_2A["weight_decay"]})')
print(f"   warmup_steps={warmup_steps}, total_steps={total_steps}")
print(f"   CrossEntropyLoss using class weights ✓")

In [ ]:
# ── Train / eval loop ─────────────────────────────────────────────────────────
def run_epoch_supervised(loader, train=True, scheduler=None):
    """
    Supervised epoch: (text, image, label) → CE loss.
    The backbone (similarity_module) runs under no_grad when frozen to
    save memory and prevent accidental gradient accumulation.
    """
    model.train() if train else model.eval()
    backbone_frozen = not next(model.similarity_module.parameters()).requires_grad

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image, labels in tqdm(
            loader, desc="Train" if train else "Eval", leave=False
        ):
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            labels = labels.to(DEVICE)

            # Encode with backbone — use no_grad when frozen to save memory
            if backbone_frozen:
                with torch.no_grad():
                    ts, is_, _ = model.similarity_module(text, image)
            else:
                ts, is_, _ = model.similarity_module(text, image)

            det, attn, skl = model.detection_module(text, image, ts, is_)
            loss = loss_ce(det, labels)

            if train:
                optim_2a.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    CFG_2A["grad_clip"],
                )
                optim_2a.step()
                if scheduler:
                    scheduler.step()

            pred = det.argmax(1)
            tot_ok += pred.eq(labels).sum().item()
            tot_n += labels.size(0)
            tot_loss += loss.item() * labels.size(0)
            all_y.extend(labels.cpu().numpy())
            all_p.extend(pred.cpu().numpy())

    return {"loss": tot_loss / tot_n, "acc": tot_ok / tot_n}, all_y, all_p


print("✅ run_epoch_supervised defined")

In [ ]:
# ── Phase 2a training loop ────────────────────────────────────────────────────
EXPERIMENT_NAME = "coolant-vifactcheck-stage2"
mlflow.set_experiment(EXPERIMENT_NAME)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"stage2a-frozen-{NUM_CLASSES}cls-{timestamp}"

best_val_loss = float("inf")
best_val_acc = 0.0
patience_counter = 0
history_2a = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

with mlflow.start_run(run_name=run_name):
    mlflow.log_params(CFG_2A)
    mlflow.log_param("num_classes", NUM_CLASSES)
    mlflow.log_param("stage1_epoch", ckpt.get("epoch", "unknown"))
    mlflow.log_param("stage1_val_loss", f"{ckpt.get('val_loss', 0):.4f}")

    print(
        f'🚀 Phase 2a — max {CFG_2A["max_epochs"]} epochs, patience={CFG_2A["patience"]}'
    )
    print(f"   Classes: {NUM_CLASSES}, Run: {run_name}\n")

    for epoch in range(1, CFG_2A["max_epochs"] + 1):
        train_m, _, _ = run_epoch_supervised(
            loaders["train"], train=True, scheduler=scheduler_2a
        )
        val_m, _, _ = run_epoch_supervised(loaders["dev"], train=False)

        history_2a["train_loss"].append(train_m["loss"])
        history_2a["train_acc"].append(train_m["acc"])
        history_2a["val_loss"].append(val_m["loss"])
        history_2a["val_acc"].append(val_m["acc"])

        mlflow.log_metrics(
            {
                "train_loss": train_m["loss"],
                "train_acc": train_m["acc"],
                "val_loss": val_m["loss"],
                "val_acc": val_m["acc"],
                "lr": optim_2a.param_groups[0]["lr"],
            },
            step=epoch,
        )

        print(
            f'Epoch [{epoch:02d}/{CFG_2A["max_epochs"]}]'
            f'  train loss={train_m["loss"]:.4f} acc={train_m["acc"]:.4f}'
            f'  val loss={val_m["loss"]:.4f} acc={val_m["acc"]:.4f}'
        )

        if val_m["loss"] < best_val_loss:
            best_val_loss = val_m["loss"]
            best_val_acc = val_m["acc"]
            patience_counter = 0
            ckpt_path = SAVE_DIR / "best_stage2a.pth"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optim_state_dict": optim_2a.state_dict(),
                    "val_loss": best_val_loss,
                    "val_acc": best_val_acc,
                    "config": CFG_2A,
                    "num_classes": NUM_CLASSES,
                    "phase": "2a",
                },
                ckpt_path,
            )
            print(f"  ✓ Saved best_stage2a.pth  (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f'  → No improvement ({patience_counter}/{CFG_2A["patience"]})')

        if patience_counter >= CFG_2A["patience"]:
            print(f"\n🛑 Early stopping at epoch {epoch}")
            break

    print(
        f"\n✅ Phase 2a done  best val_loss={best_val_loss:.4f}  val_acc={best_val_acc:.4f}"
    )

In [ ]:
# ── Test evaluation (Phase 2a) ────────────────────────────────────────────────
# Reload best 2a checkpoint
ckpt_2a = torch.load(SAVE_DIR / "best_stage2a.pth", map_location=DEVICE)
model.load_state_dict(ckpt_2a["model_state_dict"])
print(f'✅ Loaded best_stage2a.pth (epoch {ckpt_2a["epoch"]})')

test_m, test_y, test_p = run_epoch_supervised(loaders["test"], train=False)

label_names = ["Supported/Real", "Refuted/Fake", "NEI"][:NUM_CLASSES]
print(f"\n🔍 Phase 2a — Test Results")
print(f'   Loss: {test_m["loss"]:.4f}   Acc: {test_m["acc"]:.4f}')
print("\n" + classification_report(test_y, test_p, target_names=label_names, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(test_y, test_p))

---

## 🔓 Phase 2b — Unfreeze + end-to-end fine-tuning

Run this section after Phase 2a converges. Loads `best_stage2a.pth` and fine-tunes all layers at a much lower LR.


In [ ]:
# ── Phase 2b config ───────────────────────────────────────────────────────────
CFG_2B = {
    **CFG_2A,
    "lr": 5e-5,  # much lower for fine-tuning
    "max_epochs": 15,
    "patience": 5,
    "phase": "2b_full_finetune",
}
print("Phase 2b config:")
print(
    f'  lr={CFG_2B["lr"]}, max_epochs={CFG_2B["max_epochs"]}, patience={CFG_2B["patience"]}'
)

In [ ]:
# ── Load 2a checkpoint and unfreeze all ───────────────────────────────────────
ckpt_2a = torch.load(SAVE_DIR / "best_stage2a.pth", map_location=DEVICE)
model.load_state_dict(ckpt_2a["model_state_dict"])
apply_phase("2b")  # unfreeze everything

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"   Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
# ── Optimizer & scheduler for Phase 2b ───────────────────────────────────────
# Differential LR: backbone at lower LR, detection head at higher LR
optim_2b = torch.optim.AdamW(
    [
        {"params": model.similarity_module.parameters(), "lr": CFG_2B["lr"]},
        {"params": model.clip_module.parameters(), "lr": CFG_2B["lr"]},
        {"params": model.detection_module.parameters(), "lr": CFG_2B["lr"] * 5},
    ],
    weight_decay=CFG_2B["weight_decay"],
)

steps_2b = len(loaders["train"]) * CFG_2B["max_epochs"]
warmup_2b = max(1, int(0.05 * steps_2b))
scheduler_2b = torch.optim.lr_scheduler.OneCycleLR(
    optim_2b,
    max_lr=[CFG_2B["lr"], CFG_2B["lr"], CFG_2B["lr"] * 5],
    total_steps=steps_2b,
    pct_start=warmup_2b / steps_2b,
)

# Rebind optimizer used by run_epoch_supervised
optim_2a = optim_2b  # run_epoch_supervised uses optim_2a

print(f"✅ Phase 2b optimizer ready")
print(f'   backbone lr={CFG_2B["lr"]}, detection head lr={CFG_2B["lr"]*5}')

In [ ]:
# ── Phase 2b training loop ────────────────────────────────────────────────────
timestamp_2b = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name_2b = f"stage2b-fullfinetune-{NUM_CLASSES}cls-{timestamp_2b}"

best_val_loss_2b = float("inf")
best_val_acc_2b = 0.0
patience_counter = 0

with mlflow.start_run(run_name=run_name_2b):
    mlflow.log_params(CFG_2B)

    print(
        f'🚀 Phase 2b — max {CFG_2B["max_epochs"]} epochs, patience={CFG_2B["patience"]}'
    )
    print(f"   Run: {run_name_2b}\n")

    for epoch in range(1, CFG_2B["max_epochs"] + 1):
        train_m, _, _ = run_epoch_supervised(
            loaders["train"], train=True, scheduler=scheduler_2b
        )
        val_m, _, _ = run_epoch_supervised(loaders["dev"], train=False)

        mlflow.log_metrics(
            {
                "train_loss": train_m["loss"],
                "train_acc": train_m["acc"],
                "val_loss": val_m["loss"],
                "val_acc": val_m["acc"],
            },
            step=epoch,
        )

        print(
            f'Epoch [{epoch:02d}/{CFG_2B["max_epochs"]}]'
            f'  train loss={train_m["loss"]:.4f} acc={train_m["acc"]:.4f}'
            f'  val loss={val_m["loss"]:.4f} acc={val_m["acc"]:.4f}'
        )

        if val_m["loss"] < best_val_loss_2b:
            best_val_loss_2b = val_m["loss"]
            best_val_acc_2b = val_m["acc"]
            patience_counter = 0
            ckpt_path = SAVE_DIR / "best_stage2b.pth"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_loss": best_val_loss_2b,
                    "val_acc": best_val_acc_2b,
                    "config": CFG_2B,
                    "num_classes": NUM_CLASSES,
                    "phase": "2b",
                },
                ckpt_path,
            )
            print(f"  ✓ Saved best_stage2b.pth  (val_loss={best_val_loss_2b:.4f})")
        else:
            patience_counter += 1
            print(f'  → No improvement ({patience_counter}/{CFG_2B["patience"]})')

        if patience_counter >= CFG_2B["patience"]:
            print(f"\n🛑 Early stopping at epoch {epoch}")
            break

    print(
        f"\n✅ Phase 2b done  best val_loss={best_val_loss_2b:.4f}  val_acc={best_val_acc_2b:.4f}"
    )

In [ ]:
# ── Final test evaluation (Phase 2b) ─────────────────────────────────────────
ckpt_2b = torch.load(SAVE_DIR / "best_stage2b.pth", map_location=DEVICE)
model.load_state_dict(ckpt_2b["model_state_dict"])
print(f'✅ Loaded best_stage2b.pth (epoch {ckpt_2b["epoch"]})')

test_m, test_y, test_p = run_epoch_supervised(loaders["test"], train=False)

print(f"\n🔍 Phase 2b — Test Results")
print(f'   Loss: {test_m["loss"]:.4f}   Acc: {test_m["acc"]:.4f}')
print("\n" + classification_report(test_y, test_p, target_names=label_names, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(test_y, test_p))